# EMD-Transformer-BiLSTM + Weather — Prediksi AQI Jakarta
Target: Scopus Q2 | GPU T4 | Data: Open-Meteo (CAMS AQI + ERA5 weather)
Pipeline: fetch_data → EMD(12 IMF) → per-IMF Transformer-BiLSTM (vs Linear fallback) → fusi BiLSTM → RMSE/MAE/MAPE.

In [ ]:
# Kloning branch dev (kode ada di sini, bukan main)
!git clone -b dev https://github.com/madajabbar/riset-aqi-jakarta.git
%cd /content/riset-aqi-jakarta/our_model

In [ ]:
# torch sudah preinstalled di Colab
!pip install -q EMD-signal scikit-learn matplotlib pandas numpy requests

In [ ]:
# FETCH DATA (keyless, reproducible) -> ../our_data/jakarta_aqi.csv + jakarta_met.csv
!python fetch_data.py 2022-08-15 2026-10-01

import pandas as pd
aqi = pd.read_csv("../our_data/jakarta_aqi.csv", parse_dates=["datetime"])
met = pd.read_csv("../our_data/jakarta_met.csv", parse_dates=["datetime"])
print("AQI:", aqi.shape, aqi.columns.tolist())
print("MET:", met.shape, met.columns.tolist())
display(aqi.head()); display(met.head())

In [ ]:
# TRAIN our model (EMD-Transformer-BiLSTM + weather exogenous) -> RQ2
!python train.py --weather --start 2022-08-15 --end 2026-10-01 --epochs 100 2>&1 | tee train_weather.log

In [ ]:
# ABLATION base (tanpa weather)
!python train.py --start 2022-08-15 --end 2026-10-01 --epochs 100 2>&1 | tee train_base.log

In [ ]:
# HASIL
import glob
for f in sorted(glob.glob("results/*.csv")):
    print("\n===", f, "===")
    print(pd.read_csv(f).head())